In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9417, done.
remote: Counting objects: 100% (386/386), done.
remote: Compressing objects: 100% (324/324), done.
remote: Total 9417 (delta 146), reused 297 (delta 61), pack-reused 9031 (from 2)
Receiving objects: 100% (9417/9417), 1009.86 MiB | 49.85 MiB/s, done.
Resolving deltas: 100% (147/147), done.
Updating files: 100% (9022/9022), done.
/kaggle/working/O_D
ad92907 (HEAD -> main, origin/main, origin/HEAD) ver1.9.2


In [2]:
!pwd
!ls


/kaggle/working/O_D
LICENSE  object_detection.ipynb  public     requirements.txt  train.py
models	 predict.py		 README.md  results	      utils


In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 164MB/s]
/kaggle/working/O_D/train.py:523: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Class weights: [0.5563, 1.1696, 1.2632, 1.3591, 0.6519]
Epoch 001/020 | train_loss=1.3708 (cls=0.3108, reg=0.7180, ctr=0.6839) | val_loss=1.1801
Saved best checkpoint: models/best.pth (val_loss=1.1801)
Epoch 002/020 | train_loss=1.0673 (cls=0.2359, reg=0.5018, ctr=0.6592) | val_loss=1.0629
Saved best checkpoint: models/best.pth (val_loss=1.0629)
Epoch 003/020 | train_loss=0.9123 (cls=0.2078, reg=0.

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Predicted images: 1500
Saved JSON: val_predictions.json
Saved hardcase images: 100 -> results
Hardcase summary: results/hardcase_summary.json


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.483565,
  "performance_points": 15,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 3135,
  "micro_precision": 0.367464,
  "micro_recall": 0.570015,
  "per_class": {
    "person": {
      "ap": 0.581423,
      "num_ground_truth": 1074,
      "num_predictions": 2373,
      "true_positives": 720,
      "false_positives": 1653,
      "recall": 0.670391,
      "precision": 0.303413
    },
    "car": {
      "ap": 0.399248,
      "num_ground_truth": 283,
      "num_predictions": 246,
      "true_positives": 135,
      "false_positives": 111,
      "recall": 0.477032,
      "precision": 0.54878
    },
    "dog": {
      "ap": 0.636303,
      "num_ground_truth": 206,
      "num_predictions": 273,
      "true_positives": 145,
      "false_positives": 128,
      "recall": 0.703883,
      "precision": 0.531136
    },
    "cat": {
      "ap": 0.747661,
      "num_ground_truth": 176,
      "num_predictions": 228,
      "true_positives": 137,
      "f

In [7]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /kaggle/working/O_D && zip -r /kaggle/working/train.zip . -x 'public/*' 'public/**'
  adding: LICENSE (deflated 41%)
  adding: results/ (stored 0%)
  adding: results/hardcase_026_img_8e866e43628f.jpg (deflated 4%)
  adding: results/hardcase_017_img_149ecfac6be3.jpg (deflated 0%)
  adding: results/hardcase_027_img_ddf1a812fef2.jpg (deflated 0%)
  adding: results/hardcase_078_img_f968c50b0eca.jpg (deflated 0%)
  adding: results/hardcase_074_img_b995f8acd79d.jpg (deflated 0%)
  adding: results/hardcase_073_img_a5752d71ff3d.jpg (deflated 0%)
  adding: results/hardcase_020_img_20177b210046.jpg (deflated 0%)
  adding: results/hardcase_058_img_d291d4111bbe.jpg (deflated 0%)
  adding: results/hardcase_010_img_2f64659a6324.jpg (deflated 0%)
  adding: results/hardcase_025_img_70b930d7e4f9.jpg (deflated 4%)
  adding: results/hardcase_049_img_8887d945c1ff.jpg (deflated 0%)
  adding: results/hardcase_003_img_42a7f1d72dcf.jpg (deflated 0%)
  adding: results/hardcase_019_img_1d687a475fe7.jpg (defl